In [6]:
import numpy as np
import pandas as pd

#### **Importing Dataset**

In [7]:
df = pd.read_csv(r"E:\Pradhumn- DS\Deep Learning with PyTorch\data\data.csv")
df = pd.DataFrame(df)
print(df.shape)
print(df.head())

(569, 33)
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  

#### **Data Preprocessing**

In [8]:
# removing unnecessary columns
df = df.drop(columns=["id", "Unnamed: 32"])

#### **Feature and Target Selection**

In [9]:
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

#### **Label Encoding**

In [10]:
y = y.map({"B": 0,"M": 1})

#### **Train Test Split**

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

#### **Standard Scaler**

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#### **Numpy To Torch**

In [13]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)

#### **Defining Neural Network**

In [14]:
import torch
import torch.nn as nn

class MyNN(nn.Module):
    def __init__(self):
        # Parent class nn.Module ka constructor call
        super().__init__()
        
        # Define the layers of the neural network
        self.layer1 = nn.Linear(30, 64)  # Input layer to hidden layer
        
        #activation function
        self.relu = nn.ReLU()  # ReLU activation function
        
        # 2nd hidden layer
        self.layer2 = nn.Linear(64, 32)  # Hidden layer to output layer
        
        #activation function
        self.relu2 = nn.ReLU()  # ReLU activation function
        
        #32 neurons → 1 output
        self.layer3 = nn.Linear(32, 1)  # Hidden layer to output
        self.sigmoid = nn.Sigmoid()  # Sigmoid activation function for binary classification
        
    def forward(self, x):
        # Forward pass through the network
        x = self.layer1(x)  # Pass input through first layer. Apply linear transformation y = mx + b
        x = self.relu(x)    # Apply ReLU activation
        
        x = self.layer2(x)  # Pass through second layer
        x = self.relu2(x)   # Apply ReLU activation
        
        x = self.layer3(x)  # Pass through output layer
        x = self.sigmoid(x) # Apply Sigmoid activation for binary output
        
        return x  # Return the final output
        

In [15]:
model = MyNN()

loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [16]:
epochs = 500

for epoch in range(epochs):

    # Forward Pass
    y_pred = model(X_train)

    # Calculate Loss
    loss = loss_function(
        y_pred.squeeze(),
        y_train
    )

    # Remove old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch: {epoch+1}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch: 10, Loss: 0.7680
Epoch: 20, Loss: 0.7267
Epoch: 30, Loss: 0.6751
Epoch: 40, Loss: 0.6279
Epoch: 50, Loss: 0.5986
Epoch: 60, Loss: 0.5840
Epoch: 70, Loss: 0.5764
Epoch: 80, Loss: 0.5719
Epoch: 90, Loss: 0.5690
Epoch: 100, Loss: 0.5670
Epoch: 110, Loss: 0.5654
Epoch: 120, Loss: 0.5641
Epoch: 130, Loss: 0.5629
Epoch: 140, Loss: 0.5619
Epoch: 150, Loss: 0.5609
Epoch: 160, Loss: 0.5600
Epoch: 170, Loss: 0.5593
Epoch: 180, Loss: 0.5588
Epoch: 190, Loss: 0.5583
Epoch: 200, Loss: 0.5579
Epoch: 210, Loss: 0.5575
Epoch: 220, Loss: 0.5572
Epoch: 230, Loss: 0.5569
Epoch: 240, Loss: 0.5567
Epoch: 250, Loss: 0.5565
Epoch: 260, Loss: 0.5563
Epoch: 270, Loss: 0.5561
Epoch: 280, Loss: 0.5560
Epoch: 290, Loss: 0.5558
Epoch: 300, Loss: 0.5557
Epoch: 310, Loss: 0.5556
Epoch: 320, Loss: 0.5555
Epoch: 330, Loss: 0.5553
Epoch: 340, Loss: 0.5551
Epoch: 350, Loss: 0.5550
Epoch: 360, Loss: 0.5548
Epoch: 370, Loss: 0.5547
Epoch: 380, Loss: 0.5546
Epoch: 390, Loss: 0.5545
Epoch: 400, Loss: 0.5545
Epoch: 41

In [17]:
model.eval()

with torch.no_grad():

    y_pred_test = model(X_test)

    probability = torch.sigmoid(y_pred_test)

    predicted_class = (probability >= 0.5).float()

    accuracy = (
        predicted_class.squeeze() == y_test
    ).float().mean()

print("Test Accuracy:", accuracy.item())

Test Accuracy: 0.3684210479259491
